# 3 · Learned downscaling with the WeatherGenerator (O96 1° → N320 ~0.25°)

In notebook&nbsp;2 we built a **bilinear-interpolation baseline**. Now we train and run the
**WeatherGenerator** (WG) to do the same job, and check whether it beats that baseline.

We do not train a model from scratch. We start from the pretrained O96 → O96 forecast model
`tf5o4dy1` and **fine-tune only a new decoder** that reads the latent state out onto the
N320 grid:

| part | what it does | during fine-tuning |
|---|---|---|
| **encoder** | turns the coarse O96 state (`ERA5` stream) into latent tokens on a HEALPix grid | ❄️ frozen |
| **forecast engine** | processes the latent state (with `offset: 0` it does *not* move forward in time) | ❄️ frozen |
| **N320 decoder** | predicts all N320 points (`ERA5_N320` stream) from the latent state | 🔥 trained |

In this notebook we will:

1. **Launch** the decoder fine-tuning as a Slurm job.
2. **Monitor** the job and plot the **training and validation losses**.
3. Run a small **inference**: 16 samples at one forecast step (`fstep = 0`, i.e. same-time
   downscaling). This writes the predictions to a zarr/zip store.
4. **Evaluate** the predictions with WeatherGenerator's `evaluate` package.
5. **Compare** against the bilinear baseline from notebook&nbsp;2 on exactly the same samples.

> ⚙️ **Where this runs.** The notebook itself only needs a CPU (a login node or a JupyterHub
> CPU session is fine). Training and inference are submitted to the **GPU partition** with
> `sbatch`; the notebook waits for them and then reads the results. Use the
> **Python (hackathon-downscaling)** kernel.

## Configuration

Everything you might want to change is in the next cell:

- `FT_RUN_ID` — the fine-tuning run to monitor and use for inference. It defaults to an
  existing run; once you launch your own training (Part&nbsp;A), it is replaced with your new run id.
- `FSTEP` — which forecast step to downscale. `0` is same-time downscaling; `1` would be
  "forecast 6 h ahead *and* downscale".
- `NUM_SAMPLES`, `INF_START_DATE` — how many consecutive 6-hourly samples to run inference on,
  starting from which date. The default window is inside the **validation period**
  (Oct–Dec 2023), which the model never trained on.
- `CHANNELS` — the variables we score and compare with the baseline.

In [ ]:
import os
import re
import json
import time
import secrets
import string
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# ----------------------------------------------------------------------
# CONFIG — paths (defaults match the shared challenge-3 setup on Levante)
# ----------------------------------------------------------------------
CHALLENGE_DIR = Path(os.environ.get("CHALLENGE_DIR", "/work/bk1444/hackathon_2026/challenge_3"))
WGEN_REPO = Path(os.environ.get("WGEN_REPO", CHALLENGE_DIR / "WeatherGenerator"))  # shared code + .venv
LAUNCH_DIR = CHALLENGE_DIR / "launch"                                              # submit.sh / slurm_train.sh

def _default_work_dir():
    home_link = Path.home() / "work_c3"                    # created by `workshops run expect_hackathon setup c3`
    return home_link if home_link.exists() else CHALLENGE_DIR / os.environ["USER"]

WORK_DIR = Path(os.environ.get("WORK_DIR", _default_work_dir()))   # models/, results/, logs/ live here
PRIVATE_CONFIG = WORK_DIR / "paths.yml"                           # written by launch/setup_run.sh
CONFIG = "config/config_finetuning_downscaling_n320.yml"          # relative to WGEN_REPO
os.environ["WEATHERGEN_PRIVATE_CONF"] = str(PRIVATE_CONFIG)       # read by the weathergen python packages

# ----------------------------------------------------------------------
# CONFIG — experiment
# ----------------------------------------------------------------------
FT_RUN_ID = "iapznnq2"          # fine-tuning run to monitor / run inference with
MINI_EPOCH = -1                 # checkpoint to load: -1 = <run_id>_latest.chkpt
STREAM = "ERA5_N320"            # the N320 output stream (see config/streams/era5_downscaling_o96_n320/)

FSTEP = 0                       # forecast step: 0 = same-time downscaling (no forecast)
NUM_SAMPLES = 16                # number of consecutive 6-hourly samples for inference
INF_START_DATE = "2023-10-01T00:00"   # inside the validation period
INF_END_DATE = "2023-12-31T00:00"

CHANNELS = ["2t", "10u", "10v", "msl", "t_850", "z_500", "q_700"]   # variables to score
MAP_VAR = "2t"                  # variable shown in the maps

# Slurm settings for the inference job (1 GPU is plenty for a few samples)
SLURM_ACCOUNT = "bk1444"
SLURM_PARTITION = "gpu"
INF_TIME_LIMIT = "00:30:00"

for p in [WGEN_REPO, LAUNCH_DIR, WORK_DIR, PRIVATE_CONFIG]:
    print(f"{'ok ' if p.exists() else 'MISSING'}  {p}")

In [ ]:
def sh(cmd, check=False, quiet=False, **kw):
    """Run a shell command, print (and return) its output."""
    res = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    out = (res.stdout + res.stderr).strip()
    if out and not quiet:
        print(out)
    if check and res.returncode != 0:
        raise RuntimeError(f"command failed ({res.returncode}): {cmd}")
    return out


def gen_run_id():
    """Random run id in WeatherGenerator's style: one lowercase letter + 7 [a-z0-9]."""
    alnum = string.ascii_lowercase + string.digits
    return secrets.choice(string.ascii_lowercase) + "".join(secrets.choice(alnum) for _ in range(7))

## Part A — Fine-tune the N320 decoder

Training is configured by `config/config_finetuning_downscaling_n320.yml` in the shared
WeatherGenerator repo. The key settings are printed below:

- `load_chkpt` loads the pretrained `tf5o4dy1` weights.
- `freeze_modules` freezes everything whose name matches `.*encoder.*|.*forecast_engine.*`,
  so only the N320 decoder learns.
- `training_config.forecast` sets `offset: 0`, so the target window is the *same* 6-hour
  window as the input. With `num_steps: 3` the model is also trained on fsteps 1 and 2
  (downscaling + forecasting 6 h and 12 h ahead).
- `validation_config` defines the held-out validation period (Oct–Dec 2023). Validation
  runs at the end of every mini-epoch.

In [ ]:
from omegaconf import OmegaConf

cfg_ft = OmegaConf.load(WGEN_REPO / CONFIG)
for key in ["load_chkpt", "freeze_modules", "streams_directory", "training_config.forecast",
            "training_config.num_mini_epochs", "training_config.samples_per_mini_epoch",
            "validation_config.samples_per_mini_epoch", "validation_config.start_date",
            "validation_config.end_date"]:
    val = OmegaConf.select(cfg_ft, key)
    val = OmegaConf.to_container(val) if OmegaConf.is_config(val) else val
    print(f"{key:42s}: {val}")

### Launch the training job

`launch/submit.sh` prints a summary (including the new **run id**), then submits
`launch/slurm_train.sh` (2 nodes × 4 A100 GPUs, 8 h limit). Any extra arguments are forwarded
to `train`. The next cell runs the equivalent of

```bash
/work/bk1444/hackathon_2026/challenge_3/launch/submit.sh \
  --base-config /work/bk1444/hackathon_2026/challenge_3/WeatherGenerator/config/config_finetuning_downscaling_n320.yml \
  --options general.multiprocessing_method=spawn
```

Further overrides such as `training_config.learning_rate_scheduling.lr_max=1e-4` go after
the same `--options`, which takes all remaining arguments.

> ⚠️ **Always pass `general.multiprocessing_method=spawn`.** With the default `fork`, the data-loader
> workers for the first validation pass are forked from a process that already uses CUDA.
> They crash with `CUDA error: initialization error` / `DataLoader worker ... exited
> unexpectedly`, which kills the job at the end of mini-epoch&nbsp;0. `spawn` starts clean
> worker processes instead.

Set `SUBMIT_TRAINING = True` to submit. It is `False` by default so that re-running the notebook
does not queue another 8-hour job by accident. If you leave it `False`, the rest of the notebook
uses the existing run `FT_RUN_ID`.

In [ ]:
SUBMIT_TRAINING = False
TRAIN_EXTRA_ARGS = [
    "--base-config", str(WGEN_REPO / CONFIG),
    "--options", "general.multiprocessing_method=spawn",   # required, see the warning above
]
# more overrides go after the same --options, e.g.
#   "--options", "general.multiprocessing_method=spawn", "training_config.learning_rate_scheduling.lr_max=1e-4"

TRAIN_JOB_ID = None
if SUBMIT_TRAINING:
    env = dict(os.environ, WORK_DIR=str(WORK_DIR))
    out = sh([str(LAUNCH_DIR / "submit.sh"), *TRAIN_EXTRA_ARGS], env=env)
    m = re.search(r"Submitted Slurm job (\d+) for run_id (\w+)", out)
    if m is None:
        raise RuntimeError("submission failed, see output above")
    TRAIN_JOB_ID, FT_RUN_ID = m.groups()
    print(f"\n-> monitoring run {FT_RUN_ID} (Slurm job {TRAIN_JOB_ID})")
else:
    print(f"Not submitting; using existing run FT_RUN_ID = {FT_RUN_ID}")

## Part B — Monitor training

Three tools:

- `job_status(job_id)` asks Slurm about the job. `squeue` covers queued and running jobs;
  `sacct` covers finished ones.
- `tail_log(run_id)` shows the latest progress lines from `WORK_DIR/logs/train_<run_id>_<jobid>.txt`.
  It filters out the data-loader chatter.
- `plot_losses(run_id)` reads `WORK_DIR/results/<run_id>/<run_id>_train_metrics.json`. The file
  has one JSON line per logging step, tagged `"stage": "train"` or `"stage": "val"`.

**Which loss is plotted.** Training lines carry `loss_avg_mean`. Validation lines don't, but
both stages carry `LossPhysical.loss_avg`, which is the same number (the MSE in *normalised*
units, averaged over channels and forecast steps). The x-axis is `num_samples`, the number of
training samples seen so far. Validation points therefore sit at the end of each mini-epoch
(4096 samples each).

In [ ]:
_LOG_NOISE = re.compile(r"EpollSelector|Zarr event loop|iter_start|dataset \[|Sufficient available")
# lines worth showing: training progress, mini-epoch/validation markers, errors, job end
_LOG_KEEP = re.compile(r"loss =|Mini_epoch|validation \(|Error|Traceback|CANCELLED|Finished|INFERENCE")


def job_status(job_id):
    """One-line Slurm status for a job (queued/running via squeue, finished via sacct)."""
    if job_id is None:
        print("no job id"); return None
    out = sh(f"squeue -h -j {job_id} -o '%i %j %T %M/%l %R'", quiet=True)
    if not out or "Invalid job id" in out:
        out = sh(f"sacct -X -n -j {job_id} -o JobID,JobName%28,State,Elapsed,ExitCode", quiet=True)
    print(out or f"job {job_id}: no information")
    return out


def find_log(run_id, kind="train"):
    logs = sorted((WORK_DIR / "logs").glob(f"{kind}_{run_id}_*.txt"), key=os.path.getmtime)
    return logs[-1] if logs else None


def tail_log(run_id, n=15, kind="train"):
    log = find_log(run_id, kind)
    if log is None:
        print(f"no {kind} log for {run_id} yet"); return
    all_lines = [l.rstrip() for l in open(log, errors="replace") if l.strip()]
    lines = []
    for i, l in enumerate(all_lines):
        if _LOG_KEEP.search(l):
            # the validation loss value is printed on the line after "validation (<id>) :"
            lines.append(l + (" " + all_lines[i + 1].split(":", 1)[-1].strip()
                              if "validation (" in l and i + 1 < len(all_lines) else ""))
    print(f"--- {log}  (last {n} progress lines)")
    print("\n".join(lines[-n:]) if lines else "no progress lines yet (model/data still loading)")


def load_metrics(run_id):
    path = WORK_DIR / "results" / run_id / f"{run_id}_train_metrics.json"
    if not path.exists():
        print(f"no metrics yet: {path}"); return []
    return [json.loads(line) for line in open(path) if line.strip()]


def _series(rows, stage, key):
    pts = [(r["num_samples"], float(r[key])) for r in rows if r["stage"] == stage and key in r]
    return np.array(pts).T if pts else (np.array([]), np.array([]))


def plot_losses(*run_ids, channels=None, fstep=FSTEP, logy=True):
    """Train/val loss vs num_samples. Optional right panel: per-channel loss at one fstep."""
    ncols = 2 if channels else 1
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 4.5), squeeze=False)
    ax = axes[0, 0]
    for i, rid in enumerate(run_ids):
        rows = load_metrics(rid)
        x, y = _series(rows, "train", "LossPhysical.loss_avg")
        ax.plot(x, y, "-", color=f"C{i}", lw=1.5, label=f"{rid} train")
        x, y = _series(rows, "val", "LossPhysical.loss_avg")
        if len(x):
            ax.plot(x, y, "o--", color=f"C{i}", ms=6, label=f"{rid} val")
        else:
            print(f"{rid}: no validation points yet (validation runs at the end of each mini-epoch)")
    ax.set(xlabel="num_samples (training samples seen)", ylabel="loss (normalised MSE)",
           title="Training / validation loss")
    if channels:
        ax2 = axes[0, 1]
        rows = load_metrics(run_ids[0])
        for j, ch in enumerate(channels):
            key = f"LossPhysical.{STREAM}.mse.{ch}.{fstep}"
            x, y = _series(rows, "train", key)
            ax2.plot(x, y, "-", color=f"C{j}", lw=1, label=ch)
            x, y = _series(rows, "val", key)
            if len(x):
                ax2.plot(x, y, "o", color=f"C{j}", ms=5)
        ax2.set(xlabel="num_samples", ylabel="loss (normalised MSE)",
                title=f"{run_ids[0]}: per-variable loss, fstep {fstep}  (lines train, dots val)")
        ax2.legend(ncol=2, fontsize=8)
    for a in axes.flat:
        if logy:
            a.set_yscale("log")
        a.grid(alpha=0.3, which="both")
    axes[0, 0].legend()
    fig.tight_layout()
    plt.show()

In [ ]:
job_status(TRAIN_JOB_ID)
tail_log(FT_RUN_ID)

In [ ]:
plot_losses(FT_RUN_ID, channels=CHANNELS)

**Live view.** The next cell refreshes the status, the log tail and the loss plot every
`every` seconds. Interrupt the kernel (■) to stop it; the training job keeps running.

You can pass several run ids to `plot_losses`, e.g. `plot_losses("iapznnq2", "c9ebvfcj")`, to
compare runs.

In [ ]:
from IPython.display import clear_output


def monitor(run_id, job_id=None, every=120, n_updates=30):
    for _ in range(n_updates):
        clear_output(wait=True)
        print(time.strftime("%H:%M:%S"))
        state = job_status(job_id) if job_id else ""
        tail_log(run_id, n=6)
        plot_losses(run_id)
        if job_id and not re.search(r"RUNNING|PENDING|CONFIGURING", state or ""):
            print("job is no longer running — stopping the monitor"); break
        time.sleep(every)

# monitor(FT_RUN_ID, TRAIN_JOB_ID)    # uncomment to watch a running job

## Part C — Run inference on a few samples

`uv run inference` loads a trained checkpoint and runs the model over the **test
configuration**. The test config is the training config merged with `validation_config`
and then `test_config`. The downscaling config has no `test_config` block, so we set what we
need on the command line with `--options`:

| option | meaning |
|---|---|
| `test_config.samples_per_mini_epoch=16`, `test_config.output.num_samples=16` | run **and write** 16 samples |
| `test_config.shuffle=False`, `start_date`, `end_date` | consecutive 6-hourly samples from `INF_START_DATE` |
| `test_config.forecast.offset=0`, `num_steps=FSTEP+1` | produce forecast steps `0 … FSTEP`; we evaluate only `FSTEP` |
| `test_config.output.streams=[ERA5_N320]` | write only the N320 output stream |
| `data_loading.num_workers=0` | load data in the main process (fine for 16 samples, and avoids the fork/CUDA crash above) |

The predictions are written in physical units to

`WORK_DIR/results/<INF_RUN_ID>/validation_chkpt00000_rank0000.zip`

This is a zarr store inside a zip file. Its layout is `<sample>/<stream>/<fstep>/{target,prediction}`,
and each entry holds `data` (points × channels [× ensemble]), `coords` (lat, lon) and `times`.
The inference run gets its **own run id**; the resolved config is saved to
`WORK_DIR/models/<INF_RUN_ID>/`.

By default the job is submitted with `sbatch` to one A100 and the next cell waits for it.
It takes about 6 minutes for 16 samples, plus queue time. If your kernel already runs on a
GPU node, set `RUN_INFERENCE_WITH = "local"`.

> 🗂️ **No time to run inference?** Set `USE_SHARED_EXAMPLE = True`. The notebook then skips the
> job and uses a ready-made example from `/work/bk1444/hackathon_2026/challenge_3/examples/`:
> run `yeidewdz`, from the decoder `c9ebvfcj` after ~41k training samples, with the same 16
> samples at `fstep = 0`. Keep the default `FSTEP = 0` and `NUM_SAMPLES = 16` with it.

> 💾 **Disk space.** All 70 N320 channels are written for both target and prediction:
> **≈ 3.5 GB per 16 samples**. Delete old inference folders under `WORK_DIR/results/` once you
> no longer need them.

In [ ]:
USE_SHARED_EXAMPLE = False        # True: skip inference, use the shared example output (yeidewdz)
RUN_INFERENCE_WITH = "sbatch"     # "sbatch" (default) or "local" (only on a GPU node)
INF_RUN_ID = gen_run_id()

inf_options = [
    f"test_config.samples_per_mini_epoch={NUM_SAMPLES}",
    f"test_config.output.num_samples={NUM_SAMPLES}",
    "test_config.shuffle=False",
    f"test_config.start_date={INF_START_DATE}",
    f"test_config.end_date={INF_END_DATE}",
    "test_config.forecast.offset=0",
    f"test_config.forecast.num_steps={FSTEP + 1}",
    f"test_config.output.streams=[{STREAM}]",
    "data_loading.num_workers=0",
]
inf_args = [
    "--from-run-id", FT_RUN_ID,
    "--mini-epoch", str(MINI_EPOCH),
    "--run-id", INF_RUN_ID,
    f"--private-config={PRIVATE_CONFIG}",
    "--options", *inf_options,          # --options must come last (it takes all remaining args)
]
inf_cmd = "uv run inference " + " ".join(f"'{a}'" if "[" in a else a for a in inf_args)

INF_LOG = WORK_DIR / "logs" / f"inference_{INF_RUN_ID}.txt"
INF_SCRIPT = WORK_DIR / "logs" / f"inference_{INF_RUN_ID}.sh"
INF_SCRIPT.write_text(f"""#!/bin/bash
#SBATCH --job-name=wg-infer-{INF_RUN_ID}
#SBATCH --account={SLURM_ACCOUNT}
#SBATCH --partition={SLURM_PARTITION}
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=16
#SBATCH --gres=gpu:a100_80:1
#SBATCH --time={INF_TIME_LIMIT}
#SBATCH --output={INF_LOG}
#SBATCH --chdir={WORK_DIR}
set -e
cd {WGEN_REPO}          # streams_directory in the config is relative to the repo
module purge
module load nvhpc/24.7-gcc-11.2.0 git/2.43.7-gcc-11.2.0
source .venv/bin/activate
export CC=/usr/bin/gcc CXX=/usr/bin/g++ PATH=/usr/bin:$PATH
export OMP_NUM_THREADS=${{SLURM_CPUS_PER_TASK:-8}}
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
{inf_cmd} < /dev/null
echo "INFERENCE FINISHED"
""")
INF_STORE = WORK_DIR / "results" / INF_RUN_ID / "validation_chkpt00000_rank0000.zip"
INF_MODELS_DIR = WORK_DIR / "models"          # where the inference run's config is saved

if USE_SHARED_EXAMPLE:
    INF_SCRIPT.unlink()
    EXAMPLES_DIR = CHALLENGE_DIR / "examples"
    INF_RUN_ID, FT_RUN_ID, MINI_EPOCH = "yeidewdz", "c9ebvfcj", -1
    INF_STORE = EXAMPLES_DIR / "results" / INF_RUN_ID / "validation_chkpt00000_rank0000.zip"
    INF_MODELS_DIR = EXAMPLES_DIR / "models"
    print("using the shared example output — no inference job will be submitted\n")

print("inference run id :", INF_RUN_ID)
print("checkpoint       :", WORK_DIR / "models" / FT_RUN_ID / f"{FT_RUN_ID}_latest.chkpt"
      if MINI_EPOCH == -1 else f"{FT_RUN_ID} mini-epoch {MINI_EPOCH}")
print("script           :", "—" if USE_SHARED_EXAMPLE else INF_SCRIPT)
print("output           :", INF_STORE)
print("\n" + inf_cmd)

In [ ]:
def wait_for_job(job_id, poll=30, timeout=3 * 3600):
    """Block until the Slurm job leaves the queue; print state changes."""
    t0, last = time.time(), None
    while time.time() - t0 < timeout:
        state = sh(f"squeue -h -j {job_id} -o %T", quiet=True)
        if not state or "Invalid job id" in state:
            break
        if state != last:
            print(f"{time.strftime('%H:%M:%S')}  job {job_id}: {state}")
            last = state
        time.sleep(poll)
    return job_status(job_id)


if USE_SHARED_EXAMPLE:
    INF_JOB_ID = None
elif RUN_INFERENCE_WITH == "sbatch":
    out = sh(["sbatch", "--parsable", str(INF_SCRIPT)], check=True)
    INF_JOB_ID = out.strip().split(";")[0]
    wait_for_job(INF_JOB_ID)
else:
    INF_JOB_ID = None
    res = subprocess.run(["bash", "-l", str(INF_SCRIPT)], stdout=open(INF_LOG, "w"),
                         stderr=subprocess.STDOUT)
    print("exit code", res.returncode)

ok = INF_STORE.exists() and (USE_SHARED_EXAMPLE or
                             "INFERENCE FINISHED" in INF_LOG.read_text(errors="replace"))
print("\noutput written:" if ok else "\nsomething went wrong — log tail:", INF_STORE if ok else "")
if not ok:
    print("\n".join(l for l in INF_LOG.read_text(errors="replace").splitlines()
                    if not _LOG_NOISE.search(l))[-3000:])

> 💡 If the job fails, read `INF_LOG` (printed above). To reuse an earlier inference instead of
> running a new one, set `INF_RUN_ID` to its id and re-define `INF_STORE` accordingly (for runs in your own `WORK_DIR`):
> ```python
> INF_RUN_ID = "abcd1234"
> INF_STORE = WORK_DIR / "results" / INF_RUN_ID / "validation_chkpt00000_rank0000.zip"
> ```

## Part D — Look at the predictions

`weathergen.common.io.zarrio_reader` opens the store. `get_data(sample, stream, fstep)`
returns an item with `.target` (the true N320 field) and `.prediction` (the WG output). Each
has `.data`, `.coords` (lat, lon), `.times` and `.channels`.

In [ ]:
from weathergen.common.io import zarrio_reader

with zarrio_reader(INF_STORE) as zio:
    SAMPLES = sorted(int(s) for s in zio.samples)
    print("samples       :", SAMPLES)
    print("streams       :", list(zio.streams))
    print("forecast steps:", list(zio.forecast_steps))
    item = zio.get_data(SAMPLES[0], STREAM, FSTEP)
    print("channels      :", len(item.target.channels), item.target.channels[:10], "...")
    print("target shape  :", item.target.data.shape, "| prediction shape:", item.prediction.data.shape)


def load_sample(zio, sample, fstep=FSTEP, channels=CHANNELS):
    """Target and prediction for one sample, restricted to `channels` (ensemble-mean)."""
    it = zio.get_data(sample, STREAM, fstep)
    idx = [list(it.target.channels).index(c) for c in channels]
    tgt = np.asarray(it.target.data)[:, idx]
    prd = np.asarray(it.prediction.data)
    prd = (prd.mean(axis=-1) if prd.ndim == 3 else prd)[:, idx]
    lat, lon = np.asarray(it.target.coords).T
    valid_time = np.asarray(it.target.times).astype("datetime64[s]")[0]
    return dict(tgt=tgt, prd=prd, lat=lat, lon=lon, time=valid_time)


with zarrio_reader(INF_STORE) as zio:
    s0 = load_sample(zio, SAMPLES[0])
print(f"\nsample {SAMPLES[0]}: valid time {s0['time']}, {s0['lat'].size:,} N320 points")

In [ ]:
iv = CHANNELS.index(MAP_VAR)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.5), subplot_kw={"projection": ccrs.Robinson()})
vmin, vmax = np.nanpercentile(s0["tgt"][:, iv], [1, 99])
err = s0["prd"][:, iv] - s0["tgt"][:, iv]
elim = np.nanpercentile(np.abs(err), 99)
for ax, c, title, kw in [
    (axes[0], s0["tgt"][:, iv], "target (true N320)", dict(cmap="RdBu_r", vmin=vmin, vmax=vmax)),
    (axes[1], s0["prd"][:, iv], "WG prediction", dict(cmap="RdBu_r", vmin=vmin, vmax=vmax)),
    (axes[2], err, "prediction − target", dict(cmap="PuOr_r", vmin=-elim, vmax=elim)),
]:
    sc = ax.scatter(s0["lon"], s0["lat"], c=c, s=0.3, transform=ccrs.PlateCarree(), **kw)
    ax.coastlines(linewidth=0.5); ax.set_global(); ax.set_title(title)
    plt.colorbar(sc, ax=ax, shrink=0.7, orientation="horizontal", pad=0.05)
fig.suptitle(f"{MAP_VAR}, sample {SAMPLES[0]}, valid {s0['time']}, fstep {FSTEP}")
plt.show()

## Part E — Evaluate with the WeatherGenerator `evaluate` package

The `evaluate` package (`packages/evaluate` in the repo, CLI `uv run evaluate --config <yml>`)
reads inference stores, computes scores and makes standard plots. It is driven by a YAML
config (see `config/evaluate/eval_config.yml`). Here we build the same structure as a Python
dict:

- `run_ids.<INF_RUN_ID>` tells it where the store is (`results_base_dir`) and where the run's
  config lives (`model_base_dir`). Inference always writes `mini_epoch: 0`, `rank: 0`.
- `streams.ERA5_N320` selects channels, samples and forecast steps. `regrid: false` scores on
  the native N320 points.
- `evaluation.metrics` lists the scores (`rmse`, `bias`, `mae`, …).

Two ways of using it:

1. `evaluate_from_config(cfg)` does the full pipeline. It writes score files to
   `results/<INF_RUN_ID>/evaluation/` and plots (maps, histograms, score summaries) to
   `results/<INF_RUN_ID>/plots/` and `.../summary/`.
2. `calc_scores_per_stream(...)` returns the scores as an `xarray.DataArray`, so we can make
   our own plots.

> 📏 **Units.** Scores are in the variable's native units, with one exception: the evaluate
> package converts geopotential `z_*` (m²/s²) to **geopotential height** (m) by dividing by
> g = 9.80665. Notebook&nbsp;2 and Part&nbsp;F below keep m²/s².

In [ ]:
import cartopy
from omegaconf import OmegaConf

# `weathergen.evaluate` loads `hpc/platform-env.py` from the WeatherGenerator *private* repo at
# import time (only used to push scores to MLflow). The shared challenge setup has no private
# repo, so point it at a minimal stand-in that answers "unknown platform" to every query.
_default_private = WGEN_REPO.parent / "WeatherGenerator-private"
if "WEATHERGEN_PRIVATE_REPO_PATH" not in os.environ and not _default_private.is_dir():
    stub = WORK_DIR / ".weathergen_private_stub"
    (stub / "hpc").mkdir(parents=True, exist_ok=True)
    (stub / "hpc" / "platform-env.py").write_text(
        "".join(f"def {f}():\n    return None\n\n" for f in
                ["get_hpc", "get_hpc_user", "get_hpc_user_org", "get_hpc_config", "get_hpc_certificate"]))
    os.environ["WEATHERGEN_PRIVATE_REPO_PATH"] = str(stub)

# importing the evaluate package points cartopy at WORK_DIR/assets/cartopy; keep our settings
_cartopy_cfg = dict(cartopy.config)
from weathergen.evaluate.run_evaluation import evaluate_from_config
cartopy.config.update(_cartopy_cfg)

EVAL_DIR = WORK_DIR / "results" / INF_RUN_ID   # plots + score files (always in your WORK_DIR)
METRICS = ["rmse", "bias", "mae"]

eval_cfg = OmegaConf.create({
    "global_plotting_options": {"image_format": "png", "dpi_val": 120},
    "evaluation": {
        "regions": ["global"],
        "metrics": METRICS,
        "summary_plots": True,
        "summary_dir": str(EVAL_DIR / "summary"),
        "print_summary": False,
    },
    "run_ids": {
        INF_RUN_ID: {
            "label": f"WG decoder {FT_RUN_ID}",
            "mini_epoch": 0,
            "rank": 0,
            "results_base_dir": str(INF_STORE.parent),     # where the zarr/zip store is
            "model_base_dir": str(INF_MODELS_DIR),         # where the inference config is
            "runplot_base_dir": str(EVAL_DIR),
            "metrics_dir": str(EVAL_DIR / "evaluation"),
            "streams": {
                STREAM: {
                    "channels": CHANNELS,
                    "regrid": False,
                    "evaluation": {"forecast_step": [FSTEP], "sample": "all", "ensemble": "all"},
                    "plotting": {"sample": [SAMPLES[0]], "forecast_step": [FSTEP],
                                 "data_plots": ["maps", "histograms"]},
                }
            },
        }
    },
})

evaluate_from_config(eval_cfg, None)

In [ ]:
from IPython.display import Image, display

pngs = sorted(p for p in EVAL_DIR.rglob("*.png") if MAP_VAR in p.name or "summary" in str(p))
print(f"{len(pngs)} plots written under {EVAL_DIR} (showing up to 4 for {MAP_VAR} / summary):")
for p in pngs[:4]:
    print(p.relative_to(EVAL_DIR))
    display(Image(filename=str(p), width=900))

In [ ]:
from weathergen.evaluate.io.wegen_reader import WeatherGenZarrReader
from weathergen.evaluate.scores.score_orchestration import calc_scores_per_stream
from weathergen.evaluate.utils.dict_utils import parse_metric_params

reader = WeatherGenZarrReader(eval_cfg.run_ids[INF_RUN_ID], INF_RUN_ID, None)
scores = calc_scores_per_stream(reader, STREAM, ["global"], {"global": parse_metric_params(METRICS)})


def score_da(metric):
    """DataArray (sample, channel) for one metric at FSTEP (ensemble collapsed)."""
    da = scores[metric]["global"][STREAM][INF_RUN_ID]
    da = da.sel(forecast_step=FSTEP) if "forecast_step" in da.dims else da
    for dim in [d for d in da.dims if d not in ("sample", "channel")]:
        da = da.mean(dim)
    return da.sel(channel=CHANNELS)


wg_scores = {m: score_da(m) for m in METRICS}
print(f"evaluate package, {STREAM}, fstep {FSTEP}, mean over {wg_scores['rmse'].sizes['sample']} samples:")
print(f"{'channel':>8}" + "".join(f"{m:>12}" for m in METRICS))
for ch in CHANNELS:
    print(f"{ch:>8}" + "".join(f"{float(wg_scores[m].sel(channel=ch).mean()):>12.4g}" for m in METRICS))

In [ ]:
fig, axes = plt.subplots(1, len(CHANNELS), figsize=(3 * len(CHANNELS), 3), sharex=True)
for ax, ch in zip(axes, CHANNELS):
    r = wg_scores["rmse"].sel(channel=ch)
    ax.plot(r["sample"], r, "o-", ms=3)
    ax.set_title(ch); ax.set_xlabel("sample"); ax.grid(alpha=0.3)
axes[0].set_ylabel("RMSE (native units)")
fig.suptitle(f"WG RMSE per sample (evaluate package), fstep {FSTEP}")
fig.tight_layout(); plt.show()

## Part F — Compare with the bilinear baseline (notebook 2)

Now the key question: **does the learned decoder beat interpolation?**

To compare fairly, the baseline must see the same input and be scored against the same target.
For every inference sample we:

1. Read the **valid time** of the target from the store. The model's O96 **input** is the
   state at `valid time − FSTEP × 6 h`, which equals the valid time for `FSTEP = 0`.
2. Load that O96 state with `anemoi` and **bilinearly interpolate** it onto the exact N320
   points in the store, using the same method as notebook&nbsp;2.
3. Score both the WG prediction and the bilinear field against the **same target** from the
   store, using the same unweighted RMSE as notebook&nbsp;2.

All N320 points share the same source grid, so we build the triangulation **once** and reuse
the interpolation weights for every variable and sample. The result is identical to
`scipy.interpolate.griddata(method="linear")` from notebook&nbsp;2, only faster.

In [ ]:
import datetime
from anemoi.datasets import open_dataset
from scipy.spatial import Delaunay, cKDTree

SRC_NAME = "aifs-ea-an-oper-0001-mars-o96-1979-2023-6h-v8.zarr"    # coarse O96 (~1°)
TGT_NAME = "aifs-ea-an-oper-0001-mars-n320-1979-2023-6h-v8.zarr"   # fine   N320 (~0.25°)

def _default_data_root():
    for root in [os.path.expanduser("~/data_c3"), os.path.expanduser("~/data_c3/data"),
                 str(CHALLENGE_DIR / "data"), str(CHALLENGE_DIR)]:
        if os.path.exists(os.path.join(root, SRC_NAME)):
            return root
    return str(CHALLENGE_DIR / "data")

DATA_ROOT = os.environ.get("DATA_ROOT", _default_data_root())
src = open_dataset(os.path.join(DATA_ROOT, SRC_NAME))
tgt_ds = open_dataset(os.path.join(DATA_ROOT, TGT_NAME))
lon_src, lat_src = np.asarray(src.longitudes), np.asarray(src.latitudes)
src_dates = np.array(src.dates, dtype="datetime64[s]")
STEP = np.timedelta64(6, "h")


class BilinearInterpolator:
    """Linear interpolation on a Delaunay triangulation (as griddata(method="linear")).

    Source points are padded by ±360° in longitude (periodicity); query points outside the
    triangulation fall back to nearest neighbour, exactly like `interp_to_points` in notebook 2.
    """

    def __init__(self, lon_s, lat_s, lon_q, lat_q):
        n = lon_s.size
        pts = np.column_stack([np.concatenate([lon_s, lon_s - 360.0, lon_s + 360.0]),
                               np.tile(lat_s, 3)])
        q = np.column_stack([lon_q, lat_q])
        tri = Delaunay(pts)
        simplex = tri.find_simplex(q)
        inside = simplex >= 0
        T = tri.transform[simplex[inside]]
        b = np.einsum("nij,nj->ni", T[:, :2], q[inside] - T[:, 2])
        self.inside = inside
        self.weights = np.column_stack([b, 1.0 - b.sum(axis=1)])
        self.vertices = tri.simplices[simplex[inside]] % n
        self.nearest = cKDTree(pts).query(q[~inside])[1] % n

    def __call__(self, values):
        out = np.empty(self.inside.size)
        out[self.inside] = np.einsum("nj,nj->n", values[self.vertices], self.weights)
        out[~self.inside] = values[self.nearest]
        return out


def rmse(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))


t0 = time.time()
bilinear = BilinearInterpolator(lon_src, lat_src, s0["lon"], s0["lat"])
print(f"triangulation + weights for {s0['lat'].size:,} N320 points: {time.time() - t0:.1f} s "
      f"({(~bilinear.inside).sum()} points use nearest-neighbour fallback)")

In [ ]:
def o96_fields(when, channels=CHANNELS):
    """O96 source fields (points, channels) at a given time."""
    i = int(np.argmin(np.abs(src_dates - when)))
    assert src_dates[i] == when, f"{when} not in the O96 dataset"
    return np.stack([src[i][src.name_to_index[c], 0, :] for c in channels], axis=1)


nC, nS = len(CHANNELS), len(SAMPLES)
rmse_wg, rmse_bil = np.zeros((nS, nC)), np.zeros((nS, nC))
sq_wg = np.zeros(s0["lat"].size)       # per-point squared error of MAP_VAR, summed over samples
sq_bil = np.zeros(s0["lat"].size)
valid_times = []

t0 = time.time()
with zarrio_reader(INF_STORE) as zio:
    for k, s in enumerate(SAMPLES):
        d = load_sample(zio, s)
        assert np.array_equal(d["lat"], s0["lat"]) and np.array_equal(d["lon"], s0["lon"])
        f_o96 = o96_fields(d["time"] - FSTEP * STEP)          # the model's input time
        f_bil = np.stack([bilinear(f_o96[:, j]) for j in range(nC)], axis=1)
        for j in range(nC):
            rmse_wg[k, j] = rmse(d["prd"][:, j], d["tgt"][:, j])
            rmse_bil[k, j] = rmse(f_bil[:, j], d["tgt"][:, j])
        sq_wg += (d["prd"][:, iv] - d["tgt"][:, iv]) ** 2
        sq_bil += (f_bil[:, iv] - d["tgt"][:, iv]) ** 2
        valid_times.append(d["time"])
        if k == 0:
            s0["bil"], s0["o96"] = f_bil, f_o96
print(f"scored {nS} samples x {nC} variables in {time.time() - t0:.0f} s "
      f"({valid_times[0]} … {valid_times[-1]})")

# consistency check: our numpy RMSE for WG vs the evaluate package's RMSE.
# The evaluate package reports geopotential (z_*) as geopotential height, i.e. divided by g.
G = 9.80665
to_pkg_units = np.array([1 / G if c.startswith("z_") else 1.0 for c in CHANNELS])
pkg = wg_scores["rmse"].sel(sample=SAMPLES).values
print("max relative difference to the evaluate-package RMSE:",
      f"{np.nanmax(np.abs(pkg - rmse_wg * to_pkg_units) / pkg):.1e}")

### Leaderboard

`RMSE / std` divides by each variable's standard deviation from the dataset statistics, so
variables with different units can share one axis. The **skill** is `1 − RMSE_WG / RMSE_bilinear`:
positive means the WG decoder beats interpolation, and `1` would be a perfect prediction.

> 🔎 **Reading the result.** Bilinear interpolation is a strong baseline for smooth fields.
> For `2t` its error is ≈ 0.06 std, i.e. a normalised MSE of ≈ 0.004. Compare that with the
> per-variable **training loss** in Part&nbsp;B, which is in the same normalised-MSE units: the
> decoder can only win once its loss drops below that level. A checkpoint from early in
> mini-epoch&nbsp;0 is typically still well above it, and loses on every variable.

In [ ]:
std = np.array([tgt_ds.statistics["stdev"][tgt_ds.name_to_index[c]] for c in CHANNELS])
m_wg, m_bil = rmse_wg.mean(0), rmse_bil.mean(0)
skill = 1 - m_wg / m_bil
wins = (rmse_wg < rmse_bil).sum(0)

print(f"fstep {FSTEP}, mean over {nS} samples ({valid_times[0]} … {valid_times[-1]})\n")
print(f"{'variable':>8} {'RMSE bilinear':>14} {'RMSE WG':>12} {'skill':>8} {'WG wins':>9}")
for j, c in enumerate(CHANNELS):
    print(f"{c:>8} {m_bil[j]:>14.4g} {m_wg[j]:>12.4g} {skill[j]:>+8.2f} {wins[j]:>5}/{nS}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(nC)
axes[0].bar(x - 0.2, m_bil / std, 0.4, yerr=rmse_bil.std(0) / std, label="bilinear", color="0.6")
axes[0].bar(x + 0.2, m_wg / std, 0.4, yerr=rmse_wg.std(0) / std, label=f"WG ({FT_RUN_ID})", color="C0")
axes[0].set(xticks=x, xticklabels=CHANNELS, ylabel="RMSE / std", title="RMSE (normalised)")
axes[0].legend()
axes[1].bar(x, skill, color=np.where(skill > 0, "C2", "C3"))
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set(xticks=x, xticklabels=CHANNELS, ylabel="1 − RMSE_WG / RMSE_bilinear",
            title="Skill vs bilinear baseline (> 0: WG better)")
for a in axes:
    a.grid(alpha=0.3, axis="y")
fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(CHANNELS), figsize=(3 * len(CHANNELS), 3))
for j, (ax, c) in enumerate(zip(axes, CHANNELS)):
    ax.plot(rmse_bil[:, j], "o-", ms=3, color="0.5", label="bilinear")
    ax.plot(rmse_wg[:, j], "o-", ms=3, color="C0", label="WG")
    ax.set_title(c); ax.set_xlabel("sample"); ax.grid(alpha=0.3)
axes[0].set_ylabel("RMSE (native units)"); axes[0].legend(fontsize=8)
fig.suptitle(f"RMSE per sample, fstep {FSTEP}")
fig.tight_layout(); plt.show()

### Where does the learned model help?

The zoom repeats notebook&nbsp;2's Alps view, now with the WG prediction next to the bilinear
field. The global map shows, for `MAP_VAR`, the RMSE over all samples at each N320 point, as
**WG minus bilinear**. Blue areas are where the learned decoder is better.

In [ ]:
lon0, lon1, lat0, lat1 = 0, 20, 40, 52
lon_q = np.where(s0["lon"] < 0, s0["lon"] + 360, s0["lon"])    # store may use [-180, 180)
ms = (lon_src >= lon0) & (lon_src <= lon1) & (lat_src >= lat0) & (lat_src <= lat1)
mt = (lon_q >= lon0) & (lon_q <= lon1) & (s0["lat"] >= lat0) & (s0["lat"] <= lat1)
truth = s0["tgt"][mt, iv]
vmin, vmax = np.nanmin(truth), np.nanmax(truth)

panels = [
    (lon_src[ms], lat_src[ms], s0["o96"][ms, iv], 55, f"O96 input (~1°)"),
    (lon_q[mt], s0["lat"][mt], s0["bil"][mt, iv], 6, f"bilinear  RMSE {rmse(s0['bil'][mt, iv], truth):.2f}"),
    (lon_q[mt], s0["lat"][mt], s0["prd"][mt, iv], 6, f"WG  RMSE {rmse(s0['prd'][mt, iv], truth):.2f}"),
    (lon_q[mt], s0["lat"][mt], truth, 6, "true N320 (~0.25°)"),
]
fig, axes = plt.subplots(1, 4, figsize=(20, 4), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, (lo, la, c, s, title) in zip(axes, panels):
    ax.scatter(lo, la, c=c, s=s, cmap="RdBu_r", vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.6); ax.set_extent([lon0, lon1, lat0, lat1]); ax.set_title(title)
fig.suptitle(f"{MAP_VAR} over the Alps, sample {SAMPLES[0]} ({s0['time']}); RMSE over the box")
plt.show()

In [ ]:
diff = np.sqrt(sq_wg / nS) - np.sqrt(sq_bil / nS)
lim = np.nanpercentile(np.abs(diff), 99)
fig = plt.figure(figsize=(12, 5))
ax = plt.axes(projection=ccrs.Robinson())
sc = ax.scatter(s0["lon"], s0["lat"], c=diff, s=0.3, cmap="RdBu_r", vmin=-lim, vmax=lim,
                transform=ccrs.PlateCarree())
ax.coastlines(linewidth=0.5); ax.set_global()
plt.colorbar(sc, ax=ax, shrink=0.6, label="RMSE_WG − RMSE_bilinear")
ax.set_title(f"{MAP_VAR}: per-point RMSE over {nS} samples, WG minus bilinear (blue = WG better)")
plt.show()
print(f"WG better at {100 * (diff < 0).mean():.0f}% of the N320 points")

## 🧪 Your turn

1. **Train longer.** How much does the skill change between the checkpoint after one
   mini-epoch and the final one? Launch your own run in Part&nbsp;A, then re-run Parts&nbsp;C–F
   with `FT_RUN_ID` set to it.
2. **Forecast while downscaling.** Set `FSTEP = 1` or `2`: the model now predicts the N320
   state 6 h or 12 h *after* its O96 input. The baseline is then "interpolate the old state",
   which is a much weaker competitor. How does the gap change?
3. **More samples.** 16 consecutive samples cover only 4 days. Increase `NUM_SAMPLES` or move
   `INF_START_DATE` to judge how robust the leaderboard is.
4. **Harder variables.** Add `q_925`, `10u` or `skt` to `CHANNELS`. Where does interpolation
   struggle most, and does the learned decoder close the gap?
5. **Other scores.** Add metrics to `METRICS` (e.g. `"acc"`, `"psd"` for spectra, which show how
   much small-scale detail each method has) and look at the plots the evaluate package writes.

## Recap

You fine-tuned a new **N320 decoder** on top of a frozen, pretrained WeatherGenerator,
monitored its training and validation loss, ran **inference** on held-out samples, evaluated
the predictions with the **evaluate** package, and scored them against the **bilinear
baseline** on exactly the same inputs and targets.